In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ToxiPep)

This notebook processes and standardizes the **ToxiPep** dataset into a unified, machine-learning–ready format. The dataset is distributed across multiple folders corresponding to different similarity thresholds and independent evaluation sets.

- **Toxic effect / endpoint:** toxic
- **Source:** ToxiPep
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Parses peptide sequences** from all ToxiPep FASTA files, spanning multiple similarity-based subsets and an independent set.
- **Infers binary toxicity labels from sequence identifiers**:
  - identifiers containing `neg` (case-insensitive) are labeled as `0` (non-toxic),
  - all remaining sequences are labeled as `1` (toxic).
- **Standardizes the dataset schema** to the canonical format (`sequence`, `label`).
- **Performs duplicate sequence quality control**:
  - merges identical sequences with consistent labels,
  - flags sequences appearing with conflicting labels as erroneous.
- **Generates dataset-level metadata** using the centralized raw data description file.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`,
  - `detected_error_sequences.csv`,
  - `metadata.json`.

In [2]:
name_source = "ToxiPep"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df = pd.concat([
    read_fasta_doc(os.path.join(folder, file))
    for folder in [f"{PATH_INPUT}/{name_source}/0.8", 
                   f"{PATH_INPUT}/{name_source}/0.9",
                   f"{PATH_INPUT}/{name_source}/Independent set"] 
    for file in os.listdir(folder)
])

In [4]:
df = (
    df
    .assign(
        label=lambda d: d["id"]
            .str.contains("neg", case=False, na=False)
            .map({True: 0, False: 1})
    )
    [["sequence","label"]]
)
df.shape

(16712, 2)

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(9889, 2)

In [7]:
df_errors.shape

(2, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 5, 20, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from another DB, Previously published model dataset',
 'repository or server': 'https://github.com/GGCL7/ToxiPep/tree/main/Dataset',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S200103702500203X',
 'number_of_raw_sequences': 16712,
 'number_of_sequences_retained': 9889,
 'number_of_positive_sequences': 4938,
 'number_of_negative_sequences': 4951,
 'number_of_erroneous_sequences': 2,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)